In [2]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler
from torch import optim
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
import numpy as np
import os
import shutil

### Prepare the data

In [4]:
training_data_path = '/home/hbvision/mirsaid/smart-office/data/images'
extracttion_path = '/home/hbvision/mirsaid/smart-office/data/images'

os.makedirs(training_data_path, exist_ok=True)

for file in os.listdir(extracttion_path):
    class_name = file.split('.')[0].split('_')[0]
    os.makedirs(f'{training_data_path}/{class_name}', exist_ok=True)
    # Copy image to training data with real name as label
    shutil.copy(f'{extracttion_path}/{file}', f'{training_data_path}/{class_name}/{file}')

In [ ]:
data_dir = '/home/hbvision/mirsaid/smart-office/data/images'

batch_size = 32
epochs = 100
workers = 0 if os.name == 'nt' else 8

In [6]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Running on device: {}'.format(device))

Running on device: cuda:0


In [23]:
mtcnn = MTCNN(
    image_size=160, margin=0, min_face_size=20,
    thresholds=[0.6, 0.7, 0.7], factor=0.709, post_process=True,
    device=device
)

In [11]:
dataset = datasets.ImageFolder(data_dir, transform=transforms.Resize((512, 512)))
dataset.samples = [
    (p, p.replace(data_dir, data_dir + '_cropped'))
        for p, _ in dataset.samples
]
        
loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    collate_fn=training.collate_pil
)

for i, (x, y) in enumerate(loader):
    mtcnn(x, save_path=y)
    print('\rBatch {} of {}'.format(i + 1, len(loader)), end='')
    
# Remove mtcnn to reduce GPU memory usage
del mtcnn

Batch 2 of 2

In [21]:
print(f'Number of classes: {len(dataset.class_to_idx)}')

Number of classes: 31


In [13]:
resnet = InceptionResnetV1(
    classify=True,
    pretrained='vggface2',
    num_classes=len(dataset.class_to_idx)
).to(device)

In [14]:
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
scheduler = MultiStepLR(optimizer, [5, 10])

trans = transforms.Compose([
    np.float32,
    transforms.ToTensor(),
    fixed_image_standardization
])
dataset = datasets.ImageFolder(data_dir + '_cropped', transform=trans)
img_inds = np.arange(len(dataset))
np.random.shuffle(img_inds)
train_inds = img_inds[:int(0.8 * len(img_inds))]
val_inds = img_inds[int(0.8 * len(img_inds)):]

train_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(train_inds)
)
val_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(val_inds)
)

In [15]:
loss_fn = torch.nn.CrossEntropyLoss()
metrics = {
    'fps': training.BatchTimer(),
    'acc': training.accuracy
}

In [ ]:
writer = SummaryWriter()
writer.iteration, writer.interval = 0, 10

print('\n\nInitial')
print('-' * 10)
resnet.eval()
training.pass_epoch(
    resnet, loss_fn, val_loader,
    batch_metrics=metrics, show_running=True, device=device,
    writer=writer
)

for epoch in range(epochs):
    print('\nEpoch {}/{}'.format(epoch + 1, epochs))
    print('-' * 10)

    resnet.train()
    training.pass_epoch(
        resnet, loss_fn, train_loader, optimizer, scheduler,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

    resnet.eval()
    training.pass_epoch(
        resnet, loss_fn, val_loader,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

writer.close()



Initial
----------
Valid |     1/1    | loss:    3.3314 | fps:    0.1664 | acc:    0.1250   

Epoch 1/10
----------
Train |     1/1    | loss:    0.0345 | fps:  171.5336 | acc:    1.0000   
Valid |     1/1    | loss:    3.3303 | fps:   41.4959 | acc:    0.1250   

Epoch 2/10
----------
Train |     1/1    | loss:    0.0306 | fps:  178.2977 | acc:    1.0000   
Valid |     1/1    | loss:    3.3346 | fps:   48.3755 | acc:    0.1250   

Epoch 3/10
----------
Train |     1/1    | loss:    0.0302 | fps:  179.6396 | acc:    1.0000   
Valid |     1/1    | loss:    3.3463 | fps:   49.9946 | acc:    0.1250   

Epoch 4/10
----------
Train |     1/1    | loss:    0.0298 | fps:  177.9258 | acc:    1.0000   
Valid |     1/1    | loss:    3.3551 | fps:   42.4158 | acc:    0.1250   

Epoch 5/10
----------
Train |     1/1    | loss:    0.0291 | fps:  167.9159 | acc:    1.0000   
Valid |     1/1    | loss:    3.3646 | fps:   47.1696 | acc:    0.1250   

Epoch 6/10
----------
Train |     1/1    | loss: 

In [20]:
## Save the model
model_path = '/home/hbvision/mirsaid/smart-office/data/model'
os.makedirs(model_path, exist_ok=True)
torch.save(resnet.state_dict(), f'{model_path}/model.pt')


In [62]:
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
import cv2

one_test_face = '/home/hbvision/mirsaid/smart-office/body_crops/Mirsaid_20250210-144931.jpg'
img = cv2.imread(one_test_face)

face_detector = YOLO('/home/hbvision/mirsaid/smart-office/yolov8m-face.pt')
faces = face_detector(img)

for face in faces:
    for box in face.boxes:
        x1, y1, x2, y2 =  map(int, box.xyxy[0])
        img_cropped = img[y1:y2, x1:x2]

        resized_face = cv2.resize(img_cropped, (160, 160))
        resized_face = cv2.cvtColor(resized_face, cv2.COLOR_BGR2RGB)

        face_tensor = (
            torch.tensor(resized_face).permute(2, 0, 1).float().to(device)
            / 255.0
        )
        face_tensor = face_tensor.unsqueeze(0)

        # Classify the face using our model
        result = resnet(face_tensor)


0: 960x576 1 face, 27.0ms
Speed: 1.6ms preprocess, 27.0ms inference, 0.5ms postprocess per image at shape (1, 3, 960, 576)


In [63]:
result.argmax(dim=1).item()

20

In [65]:
dataset.class_to_idx

{'Abdulhamid': 0,
 'Abdulloh': 1,
 'Arsen': 2,
 'Asadbek': 3,
 'Azamat': 4,
 'Bahodir': 5,
 'BahodirNematjonov': 6,
 'Batkhuu': 7,
 'Dilshod': 8,
 'Diyorjon': 9,
 'Elyor': 10,
 'Gofurjon': 11,
 'Humoyun': 12,
 'Javokhir': 13,
 'Jumabek': 14,
 'Kamoliddin': 15,
 'Madamin': 16,
 'Maruf': 17,
 'Mirjalol': 18,
 'Mironshoh': 19,
 'Mirsaid': 20,
 'Murtazo': 21,
 'Murtazokhon': 22,
 'Otabek': 23,
 'Oybek': 24,
 'Saidazizkhon': 25,
 'Sarvar': 26,
 'Shohruh': 27,
 'Sokhibjon': 28,
 'Tilov': 29,
 'Uktam': 30}